In [1]:
!nvidia-smi

Sat Jun  6 06:24:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.124.06             Driver Version: 570.124.06     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 6000 Ada Gene...    On  |   00000000:E1:00.0 Off |                  Off |
| 30%   26C    P8             21W /  300W |       2MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers==4.40.1 tokenizers==0.19.1 timm==0.9.10 accelerate


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
import transformers, timm
print("transformers:", transformers.__version__)  # 要 4.40.1
print("timm:", timm.__version__)                    # 要 0.9.10

transformers: 4.40.1
timm: 0.9.10


In [4]:
from transformers import AutoModelForVision2Seq, AutoProcessor
import torch

processor = AutoProcessor.from_pretrained("openvla/openvla-7b", trust_remote_code=True)

model = AutoModelForVision2Seq.from_pretrained(
    "openvla/openvla-7b",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,   # 半精度,~14GB,48GB 卡轻松装下
    low_cpu_mem_usage=True,
).to("cuda")                       # 这次可以正常 .to(),因为没量化

print("模型设备:", next(model.parameters()).device)
print("模型加载完成 ✅")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


processor_config.json:   0%|          | 0.00/130 [00:00<?, ?B/s]

processing_prismatic.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/openvla/openvla-7b:
- processing_prismatic.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


preprocessor_config.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/6.97G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.16G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

模型设备: cuda:0
模型加载完成 ✅


In [6]:
from PIL import Image
import requests

# 拿一张示例机械臂场景图(OpenVLA 训练分布内的视角)
url = "https://raw.githubusercontent.com/openvla/openvla/main/prismatic/vla/datasets/rlds/oxe/materials/widowx.jpg"
try:
    image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
except Exception:
    # 备用:用一张随机彩色图占位,先验证推理管线能跑通
    import numpy as np
    image = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))

# 语言指令——OpenVLA 要求这个固定格式
instruction = "pick up the object"
prompt = f"In: What action should the robot take to {instruction}?\nOut:"

# 重新构造 inputs,确保 input_ids / pixel_values 是正确的 tensor
inputs = processor(prompt, image)
inputs = {k: v.to("cuda", dtype=torch.bfloat16) if v.dtype == torch.float32 else v.to("cuda")
          for k, v in inputs.items()}

action = model.predict_action(**inputs, unnorm_key="bridge_orig", do_sample=False)

print("指令:", instruction)
print("输出动作向量 (7维):", action)
print("含义: [Δx, Δy, Δz, Δroll, Δpitch, Δyaw, gripper]")

指令: pick up the object
输出动作向量 (7维): [-0.00289288  0.02482914  0.00443561  0.00880144 -0.0121683   0.06601701
  0.        ]
含义: [Δx, Δy, Δz, Δroll, Δpitch, Δyaw, gripper]
